# 10 · EdNet-KT1: data source and pre-processing

Supports thesis Section 4.2 (*Datasets*, EdNet-KT1) and Appendix *EdNet run*.
It documents, with the numbers recomputed from the files, how the public EdNet-KT1 log becomes the answer-event table and the pyKT inputs the models are trained on:

1. the raw data (KT1 user files and the question table);
2. mapping to the project's answer-event schema;
3. sampling: the filter funnel and the nested cohorts;
4. pyKT preparation: chronological sequences, truncation, learner-level split, leakage check.

Nothing here trains a model. Real learners, but benchmark only: no weight, KC or state reaches Moodle.

**Where to run.** Locally (recommended): `.venv-kt` with `requirements-analysis.txt`, started from the repository; no GPU needed.
**On Colab (CPU runtime is enough):** upload `ednet_colab_bundle.zip` (built by `knowledge_tracing/make_colab_bundle.py`) to `MyDrive/ednet_kt1/`; the training runs already live in `MyDrive/ednet_kt1/data/benchmarks/` (for the English run, upload `data/benchmarks/english_kt/` there too). Then *Runtime → Run all*: the first cell mounts Drive, unpacks the bundle and links the benchmark folder. Figures and tables are written to `data/benchmarks/report/` (on Drive when on Colab).

In [1]:
import sys, os
if "google.colab" in sys.modules:
    !pip install -q seaborn
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "knowledge_tracing").is_dir() and _root.parent != _root:
    _root = _root.parent
if "google.colab" in sys.modules and not (_root / "knowledge_tracing").is_dir():
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    import time as _t
    with zipfile.ZipFile("/content/drive/MyDrive/ednet_kt1/ednet_colab_bundle.zip") as zf:
        for _i in zf.infolist():   # keep the zip's timestamps: the runner compares mtimes
            _p = zf.extract(_i, "/content/ednet")
            _m = _t.mktime(_i.date_time + (0, 0, -1))
            os.utime(_p, (_m, _m))
    _root = Path("/content/ednet")
sys.path[:0] = [str(_root), str(_root / "knowledge_tracing")]
import report_utils as ru
ROOT = ru.setup()
ru.style()
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
pd.set_option("display.width", 160, "display.max_columns", 30)
print("repository:", ROOT)

repository: /home/rene/MicroLearningThesis


## 1. Raw data
EdNet-KT1 holds one CSV per user (`timestamp, solving_id, question_id, user_answer, elapsed_time`). `questions.csv` gives each question's correct option, TOEIC part (1–7) and tags. The 1.2 GB archive is read in place and never extracted; its hash is in the manifest.

In [2]:
EDNET = Path("data/external/ednet")
manifest = json.loads((EDNET / "ednet_manifest.json").read_text())
print(json.dumps({k: v for k, v in manifest.items() if k not in ("cohorts", "kt1_zip", "questions")}, indent=1))
questions = pd.read_csv(EDNET / "questions.csv")
display(questions.head())

{
 "dataset": "EdNet-KT1 (Choi et al., AIED 2020), CC BY-NC 4.0",
 "kt1_zip_sha256": "0d13933f90201c5101c7fe8659e44474fa049e3fb93181a8ba6fb3e63267b535",
 "questions_sha256": "137d2936b746c98e1586ef134bf9f9bc4b5a878c96b88e630b2241fe9f15d3cd",
 "seed": 20240116,
 "min_answers": 20,
 "max_answers": 200,
 "kc_rule": "first tag of questions.tags; tag -1 -> untagged_p<part>",
 "kc_dimension_rule": "most frequent TOEIC part of the KC's questions",
 "nested_cohorts": true,
 "sampling": {
  "user_files": 784309,
  "files_read": 122239,
  "ineligible_short": 72239,
  "unknown_question_rows": 0,
  "eligible_taken": 50000
 }
}


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439


In [3]:
raw_zip = EDNET / "EdNet-KT1.zip"
if raw_zip.exists():   # one user file, to show the raw format (local only; Colab has no archive)
    import zipfile
    with zipfile.ZipFile(raw_zip) as zf:
        member = next(n for n in zf.namelist() if n.endswith(".csv"))
        with zf.open(member) as f:
            display(pd.read_csv(f).head(8))
        print(member)
else:
    print("EdNet-KT1.zip not present here; the cohort files below are what the experiments use.")

,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1535196434651,1,q8098,b,20000
1,1535196459648,2,q8074,c,22000
2,1535196482553,3,q176,b,21000
3,1564677167247,4,q1279,c,17000
4,1564677196590,5,q6363,d,27000
5,1564677228860,6,q3670,c,29000
6,1564677273528,7,q3788,b,42000
7,1564677307843,8,q3815,b,31000


KT1/u246317.csv


### Question table: parts and tags
Each question can carry several tags. The adapter uses the **first** tag as the KC (one KC per question, as pyKT's single-concept setting needs); questions with tag `-1` get one KC per part.

In [4]:
q = questions.copy()
q["n_tags"] = q["tags"].astype(str).str.split(";").str.len()
q["first_tag"] = q["tags"].astype(str).str.split(";").str[0]
qt = pd.DataFrame({
    "questions": q.groupby("part").size(),
    "distinct first tags": q.groupby("part")["first_tag"].nunique(),
    "share with >1 tag": q.groupby("part")["n_tags"].apply(lambda s: (s > 1).mean()).round(3),
}).reset_index().rename(columns={"part": "TOEIC part"})
display(qt)
print(f"questions: {len(q):,} | with more than one tag: {(q['n_tags'] > 1).mean():.1%} | untagged (-1): {(q['first_tag'] == '-1').mean():.2%}")
ru.save_table(qt, "ednet_question_parts", "EdNet-KT1 question table by TOEIC part",
              "tab:ednet-question-parts", digits=3)

,TOEIC part,questions,distinct first tags,share with >1 tag
0,1,643,22,0.998
1,2,1662,18,1.000
2,3,1266,12,1.000
3,4,1158,15,0.995
4,5,5703,73,0.000
5,6,1335,64,0.125
6,7,1402,9,0.827


questions: 13,169 | with more than one tag: 45.9% | untagged (-1): 6.05%
table  -> data/benchmarks/report/tables/ednet_question_parts.tex


## 2. Mapping to the answer-event schema
| Answer-event field | EdNet-KT1 source |
|---|---|
| `learner_id` | user file name (`u<id>`) |
| `item_id` | `question_id` |
| `correct` | 1 if `user_answer` equals the question's `correct_answer`, else 0 |
| `kc_id` | first tag of the question (`untagged_p<part>` if none) |
| `kc_dimension` | most frequent TOEIC part among the KC's questions |
| `timestamp` | `timestamp` (ms) |
| `response_time_s` | `elapsed_time` / 1000 |
| order | time order within the learner; first 200 answers kept |

In [5]:
ev = pd.read_csv(EDNET / "kt1_events_5000.csv")
display(ev.head())
print(ev.dtypes.to_string())

,learner_id,event_index,item_id,kc_id,kc_dimension,event_type,timestamp,response_time_s,correct
0,u577866,0,q3787,tag106,part5,answer,1556621737,31.0,0
1,u577866,1,q8457,tag74,part5,answer,1556621770,30.0,0
2,u577866,2,q6168,tag76,part5,answer,1556621803,28.0,0
3,u577866,3,q5255,tag90,part5,answer,1556621842,34.0,1
4,u577866,4,q9580,tag119,part5,answer,1556621896,49.0,1


learner_id             str
event_index          int64
item_id                str
kc_id                  str
kc_dimension           str
event_type             str
timestamp            int64
response_time_s    float64
correct              int64


## 3. Sampling: filter funnel and nested cohorts
User files are shuffled with a fixed seed and read in that order; a user is eligible with at least 20 answers. The first *N* eligible users form the cohort of size *N*, so every cohort contains the smaller ones.

In [6]:
s = manifest["sampling"]
funnel = pd.DataFrame([
    ["user files in EdNet-KT1", s["user_files"]],
    ["files read (shuffled order, seed %d)" % manifest["seed"], s["files_read"]],
    ["excluded: fewer than %d answers" % manifest["min_answers"], s["ineligible_short"]],
    ["eligible users taken", s["eligible_taken"]],
], columns=["step", "users"])
display(funnel)
print(f"eligible share among files read: {s['eligible_taken'] / s['files_read']:.1%}")
ru.save_table(funnel, "ednet_funnel", "EdNet-KT1 sampling funnel", "tab:ednet-funnel", align="lr")

,step,users
0,user files in EdNet-KT1,784309
1,"files read (shuffled order, seed 20240116)",122239
2,excluded: fewer than 20 answers,72239
3,eligible users taken,50000


eligible share among files read: 40.9%
table  -> data/benchmarks/report/tables/ednet_funnel.tex


In [7]:
coh = pd.DataFrame([{**{"learners": int(k)}, **v} for k, v in manifest["cohorts"].items()])
coh = coh.sort_values("learners")
cohorts = coh[["learners", "answers", "items", "kcs", "positive_rate", "mean_seq_len", "items_multi_tag_share"]].rename(
    columns={"kcs": "KCs", "positive_rate": "correct rate", "mean_seq_len": "answers per learner",
             "items_multi_tag_share": "items >1 tag"})
display(cohorts)
ru.save_table(cohorts, "ednet_cohorts", "EdNet-KT1 nested cohorts (from the sampling manifest)", "tab:ednet-cohort-sizes", digits=3, size="footnotesize")

,learners,answers,items,KCs,correct rate,answers per learner,items >1 tag
0,300,28437,6748,141,0.6044,94.79,0.4630
1,1000,91599,9431,142,0.6040,91.60,0.4511
2,2000,180226,10530,143,0.6048,90.11,0.4560
3,5000,452335,11423,143,0.6026,90.47,0.4694
4,10000,905526,11783,143,0.6022,90.55,0.4770
5,20000,1814348,11955,143,0.6029,90.72,0.4807
6,50000,4534633,12119,143,0.6038,90.69,0.4849


table  -> data/benchmarks/report/tables/ednet_cohorts.tex


In [8]:
# Nesting check: every cohort's learners are a subset of the next larger cohort's.
sizes = sorted(int(k) for k in manifest["cohorts"])
prev = None
for n in sizes:
    ids = set(pd.read_csv(EDNET / f"kt1_events_{n}.csv", usecols=["learner_id"])["learner_id"])
    assert len(ids) == n, (n, len(ids))
    if prev is not None:
        assert prev <= ids, f"cohort {n} does not contain the previous cohort"
    prev = ids
print("nested:", " ⊂ ".join(f"{n:,}" for n in sizes))

nested: 300 ⊂ 1,000 ⊂ 2,000 ⊂ 5,000 ⊂ 10,000 ⊂ 20,000 ⊂ 50,000


## 4. pyKT preparation
`prepare_data.py` keeps answer events with a binary outcome, orders each learner's answers by time, builds the question and KC index maps, and assigns **learners** (not answers) to train/validation/test (70/10/20, seed 20240116). `init_pykt_dataset.py` writes pyKT's sequence files: the validation fold is the model-selection fold, the test fold is used once, in evaluation.

In [9]:
BENCH = ru.ednet_bench_dir()
print("benchmark folder:", BENCH)
rows = []
for n in sizes:
    proc = BENCH / f"ednet_kt1_{n}" / "kt" / "processed"
    if not (proc / "split_assignments.csv").exists():
        continue
    sp = pd.read_csv(proc / "split_assignments.csv")
    assert sp["learner_id"].is_unique, "a learner appears twice"
    c = sp["split"].value_counts()
    rows.append({"learners": n, "train": c.get("train", 0), "valid": c.get("valid", 0), "test": c.get("test", 0),
                 "at the 200-answer cap": int((sp["seq_len"] >= 200).sum())})
splits = pd.DataFrame(rows)
display(splits)
ru.save_table(splits, "ednet_splits",
              "Learner-level split of each EdNet-KT1 cohort (70/10/20)", "tab:ednet-splits")

benchmark folder: data/colab/ednet_kt1_results/benchmarks


,learners,train,valid,test,at the 200-answer cap
0,300,210,30,60,75
1,1000,700,100,200,237
2,2000,1400,200,400,463
3,5000,3500,500,1000,1179
4,10000,7000,1000,2000,2391
5,20000,14000,2000,4000,4832


table  -> data/benchmarks/report/tables/ednet_splits.tex


In [10]:
pm = json.loads((BENCH / "ednet_kt1_5000/kt/processed/prepare_data_manifest.json").read_text())
print(json.dumps(pm, indent=1)[:2500])

{
 "input": "data/external/ednet/kt1_events_5000.csv",
 "dataset_name": "ednet_kt1",
 "question_mapping": null,
 "mapping_coverage": null,
 "seed": 20240116,
 "split_ratios": {
  "train": 0.7,
  "valid": 0.1,
  "test": 0.2
 },
 "fold_code": {
  "train": 0,
  "valid": 1,
  "test": 2
 },
 "max_seq_len": 200,
 "timestamp_unit": "s",
 "checks": {},
 "filter_info": {
  "event_type_counts": {
   "answer": 452335
  },
  "answer_rows_before_cleaning": 452335,
  "dropped_non_binary_correct": 0
 },
 "order_info": {
  "timestamp_order_violations": 0
 },
 "sequence_info": {
  "learners_truncated": 0,
  "max_seq_len_observed": 200
 },
 "split_stats": {
  "train": {
   "learners": 3500,
   "answers": 316033,
   "positive_rate": 0.6025,
   "mean_seq_len": 90.3
  },
  "valid": {
   "learners": 500,
   "answers": 46623,
   "positive_rate": 0.6043,
   "mean_seq_len": 93.25
  },
  "test": {
   "learners": 1000,
   "answers": 89679,
   "positive_rate": 0.6021,
   "mean_seq_len": 89.68
  }
 }
}
